In [ ]:
import sisl
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

%matplotlib inline

## Bond transmissions

A common visualization technique for transport calculations is the so-called *bond currents* or *bond transmissions*. The nomenclature is somethat disperse, and some don't really distinguish them. But in TBtrans we try to make it clear how they differentiate.

The bond-transmission is a post-calculation form *orbital tranmsissions*:
$$
            \mathbf T_{ij}(E) = i [
            (\mathbf H_{ji} - E\mathbf S_{ji}) \mathbf A_{ij}(E)
            -
            (\mathbf H_{ij} - E\mathbf S_{ij}) \mathbf A_{ji}(E)],
$$
where here, indices $ij$ are orbital indices. An orbital transmission is a physical quantity where its the transport probability that an electron will jump from $i\to j$. They obey the Kirchhoffs laws (there cannot be any accummulation of charge at any site).  
When calculating the bond transmissions, we are summing all orbital tranmsissions per atomic pair:
$$
          \mathbf  T_{IJ}(E) = \sum_{i\in I}\sum_{j\in J} \mathbf T_{ij}(E)
$$
where here $IJ$ are atomic indices.

Finally, we can calculate the full bond-current by integrating over the bias window (meaning that for a $0V$ calculation, the bond-current is exactly 0).
$$
\mathbf I_{IJ} = \sum_{i\in I}\sum_{j\in J} \int_{-V/2}^{V/2}\mathbf T_{ij}\,\mathrm dE
$$

These quantities are sparse matrices with the same size as the Hamiltonian matrix.  
They have certain properties:
- they are symmetric, meaning that $\mathbf T_{ij} = -\mathbf T_{ji}$.
- you can take any cross section of the device by splitting the device into 2, region $A$ and $B$. And the resulting orbital-transmission sum will be equal to the transmission:
  $$
   T = \left|\sum_{\substack{i\in A\\j\in B}} \mathbf T_{ij}\right|
  $$
  Since the bond-transmissions is nothing but a sum of the orbital-transmissions, then this also holds for the bond-transmissions.  
  **NOTE**: Only for a 2-terminal system.

A last detail is the usage of *atomic currents*. These represents the amount of transmission running *through* an atom. As such it is an *atomic* quantity that kind of tells how important that atom is for redistributing electrons. Remark, that it does not contain any information on whether the its a transmitted or reflected electron. The atomic current is defined as:

$$
\begin{aligned}
  T_I^{|a|} &=\frac 12 \sum_{\{J\}} \Big| \sum_{i\in I}\sum_{j\in J} \mathbf T_{ij} \Big|
  \\
  T_I^{|o|} &=\frac 12 \sum_{i\in I}\sum_{j\in\{J\}} \big| \mathbf T_{ij} \big|
\end{aligned}
$$

### Square lattice

In this exercise we will use a square lattice with a small defect (to get some scattering) and do a post-processing on the bond-transmissionsT

In [ ]:
sc = sisl.geom.sc(1, atoms=sisl.Atom(1, R=1.01)).add_vacuum(10, axis=2)
elec = sc.tile(15, axis=0).tile(2, 1)
H_elec = sisl.Hamiltonian(elec)
r = (0.1, 1.01)
t = (0, -1.0)
H_elec.construct([r, t])
H_elec.write("ELEC.nc")

In [ ]:
# %solution clear
H_tb = sisl.Hamiltonian(sc)
H_tb.construct([r, t])
band = sisl.BandStructure(
    H_tb,
    [[0.0, 0.0], [0.5, 0], [0.5, 0.5]],
    301,
    [r"$\Gamma$", "M", "M1"],
)
eigs = band.apply.array.eigh()

# Retrieve the tick-marks and the linear k points
xtick, xtick_label = band.lineartick()
lk = band.lineark()
plt.plot(lk, eigs)

plt.ylabel("Eigenspectrum [eV]")
plt.gca().xaxis.set_ticks(xtick)
plt.gca().set_xticklabels(xtick_label)

# Also plot x-major lines at the ticks
ymin, ymax = plt.gca().get_ylim()
for tick in xtick:
    plt.plot([tick, tick], [ymin, ymax], "k")

In [ ]:
H_device = H_elec.tile(7, 1)
cxyz = H_device.geometry.center(what="xyz") + 0.1
H_device = H_device.remove(H_device.geometry.close(cxyz, R=1.3))
H_device.write("DEVICE.nc")
H_device.geometry.plot(axes="xy")

In [ ]:
%%bash
#%solution clear
rm -f siesta.TBT.nc
tbtrans RUN.fdf > TBT.out

In [ ]:
tbt = sisl.get_sile("siesta.TBT.nc")

In [ ]:
_, axs = plt.subplots(1, 2, figsize=(12, 7))
axs[0].plot(tbt.E, tbt.ADOS(0))
axs[0].set_xlabel("Energy [eV]")
axs[0].set_ylabel("ADOS[Left]")
axs[0].set_ylim([0, None])
axs[1].plot(tbt.E, tbt.transmission(0, 1))
axs[1].set_xlabel("Energy [eV]")
axs[1].set_ylabel("T")
axs[1].set_ylim([0, None]);

In [ ]:
geom = tbt.geometry


def plot_atoms(ax, geom, atomic_t):
    scale = 100 / atomic_t.max()
    ax.scatter(
        geom.xyz[:, 0], geom.xyz[:, 1], s=atomic_t * scale, c="lightgrey", zorder=1
    )
    ax.set_aspect("equal")
    ax.set_xlabel("x [Ang]")
    ax.set_ylabel("y [Ang]")


def plot_bond(ax, geom, Jij, scale=0.9):
    # Bond transmissions are J_IJ (sparse, atoms x supercell atoms).
    # Draw an arrow for each positive (outgoing) bond-transmission I -> J.
    Jij = Jij.tocoo()
    idx = Jij.data > 0
    ia, ja, J = Jij.row[idx], Jij.col[idx], Jij.data[idx]
    d = (geom.axyz(ja) - geom.xyz[ia]) * scale
    q = ax.quiver(
        geom.xyz[ia, 0],
        geom.xyz[ia, 1],
        d[:, 0],
        d[:, 1],
        J,
        cmap="viridis",
        norm=plt.Normalize(0, J.max()),
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.003,
        headwidth=3,
        zorder=2,
    )
    return q


def plot_vector(ax, geom, Ji):
    # Vector transmissions are J_I (atoms x 3), i.e. the sum of all outgoing
    # bond-transmissions from each atom.
    # Renormalize so the largest arrow has the length of a bond.
    norm = np.linalg.norm(Ji, axis=1)
    scale = 1.1 / norm.max()
    q = ax.quiver(
        geom.xyz[:, 0],
        geom.xyz[:, 1],
        Ji[:, 0] * scale,
        Ji[:, 1] * scale,
        norm,
        cmap="viridis",
        norm=plt.Normalize(0, norm.max()),
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.003,
        headwidth=3,
        zorder=2,
    )
    return q


def plot_transmission(iE, bond):
    E = tbt.E[iE]
    fig, ax = plt.subplots(figsize=(8, 8))
    atomic_T = tbt.atom_transmission(E)
    plot_atoms(ax, geom, atomic_T)
    if bond:
        q = plot_bond(ax, geom, tbt.bond_transmission(E, what="out"))
        ax.set_title(f"Bond transmissions at E = {E:.3f} eV")
    else:
        q = plot_vector(ax, geom, tbt.vector_transmission(E, what="out"))
        ax.set_title(f"Vector transmissions at E = {E:.3f} eV")
    fig.colorbar(q, ax=ax, shrink=0.7)
    plt.show()


slider = widgets.IntSlider(
    value=len(tbt.E) // 2,
    min=0,
    max=len(tbt.E) - 1,
    description="Energy index",
    continuous_update=False,
)

bond_current = widgets.ToggleButton(False, description="Bond transmissions?")

out = widgets.interactive_output(
    plot_transmission, {"iE": slider, "bond": bond_current}
)
display(widgets.HBox([slider, bond_current]), out)

## Exercises

- Go back to device creation, and make a pristine square lattice (no hole). Redo the calculation in another directory.
  Pick a single energy-point, and do the bond and vector transmissions for that energy point.
- Plot the bond and vector transmissions for the $\Gamma$ point (hint: `kindex`).
- Plot the bond and vector transmissions for any $\mathbf k\neq\Gamma$ and then also for $-\mathbf k$.
  What should this tell you about symmetries?

In [ ]:
%%bash
#%solution clean
mkdir -p square
cp RUN.fdf square/

In [ ]:
# %solution clean
H_elec.write("square/ELEC.nc")
H_sq = H_elec.tile(7, 1)
H_sq.write("square/DEVICE.nc")
H_sq.geometry.plot(axes="xy")

In [ ]:
%%bash
#%solution clean
cd square
rm -f siesta.TBT.nc
tbtrans RUN.fdf > TBT.out
cd ..

In [ ]:
# %solution start
tbt_sq = sisl.get_sile("square/siesta.TBT.nc")
geom_sq = tbt_sq.geometry
print(geom_sq)
ik_G = tbt_sq.kindex([0] * 3)
ik_k = tbt_sq.kindex([0, 0.02, 0])
ik_mk = tbt_sq.kindex(-tbt_sq.k[ik_k])

In [ ]:
# %solution clear
plt.plot(tbt_sq.E, tbt_sq.transmission(0, 1, kavg=ik_G))

In [ ]:
# %solution clear
E = 0.495
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

atomic_T = tbt_sq.atom_transmission(E, kavg=ik_G)
plot_atoms(axs[0], geom_sq, atomic_T)
q0 = plot_bond(axs[0], geom_sq, tbt_sq.bond_transmission(E, kavg=ik_G, what="out"))
axs[0].set_title(f"Bond transmissions at E = {E:.3f} eV k={tbt_sq.k[ik_G]}")
fig.colorbar(q0, ax=axs[0], shrink=0.7)

plot_atoms(axs[1], geom_sq, atomic_T)
q1 = plot_vector(axs[1], geom_sq, tbt_sq.vector_transmission(E, kavg=ik_G, what="out"))
axs[1].set_title(f"Vector transmissions at E = {E:.3f} eV")
fig.colorbar(q1, ax=axs[1], shrink=0.7)

In [ ]:
# %solution clear
E = 0.495
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

atomic_T = tbt_sq.atom_transmission(E, kavg=ik_k)
plot_atoms(axs[0], geom_sq, atomic_T)
q0 = plot_bond(axs[0], geom_sq, tbt_sq.bond_transmission(E, kavg=ik_k, what="out"))
axs[0].set_title(f"Bond transmissions at E = {E:.3f} eV k={tbt_sq.k[ik_k]}")
fig.colorbar(q0, ax=axs[0], shrink=0.7)

atomic_T = tbt_sq.atom_transmission(E, kavg=ik_mk)
plot_atoms(axs[1], geom_sq, atomic_T)
q0 = plot_bond(axs[1], geom_sq, tbt_sq.bond_transmission(E, kavg=ik_mk, what="out"))
axs[1].set_title(f"Bond transmissions at E = {E:.3f} eV k={tbt_sq.k[ik_mk]}")
fig.colorbar(q0, ax=axs[1], shrink=0.7)

In [ ]:
# %solution clear
E = 0.495
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

atomic_T = tbt_sq.atom_transmission(E, kavg=ik_k)
plot_atoms(axs[0], geom_sq, atomic_T)
q0 = plot_vector(axs[0], geom_sq, tbt_sq.vector_transmission(E, kavg=ik_k, what="out"))
axs[1].set_title(f"Vector transmissions at E = {E:.3f} eV k={tbt_sq.k[ik_k]}")
fig.colorbar(q0, ax=axs[0], shrink=0.7)

atomic_T = tbt_sq.atom_transmission(E, kavg=ik_mk)
plot_atoms(axs[1], geom_sq, atomic_T)
q1 = plot_vector(axs[1], geom_sq, tbt_sq.vector_transmission(E, kavg=ik_mk, what="out"))
axs[1].set_title(f"Vector transmissions at E = {E:.3f} eV k={tbt_sq.k[ik_mk]}")
fig.colorbar(q1, ax=axs[1], shrink=0.7)

## Learned objectives

- Understanding of the bond transmissions
- Using atomic currents
- Ensuring that momentum is considered when doing bond transmissions
- Plotting specific $\mathbf k$-point details.